# Credit Analysis
    a.	Preprocess and clean if necessary.
    b.	Build a model predicting “Risk”. 
    c.	Remember to comment your code and give rationales for models, algorithms, and approaches. 


![image.png](attachment:image.png)![image-2.png](attachment:image-2.png)![image-3.png](attachment:image-3.png)![image-4.png](attachment:image-4.png)

## Import Packages

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# %matplotlib inline

## Load data

In [0]:
# import pyodbc
# import urllib
# import sqlalchemy


# '''connect to datahub'''

# params_datahub = urllib.parse.quote_plus("DRIVER={SQL Server Native Client 11.0};"
#                                  "SERVER=localhost\SQLEXPRESS;"
#                                  "DATABASE=datahub;"
#                                  "UID=sa;"
#                                  "PWD=user1")

# engine_datahub = sqlalchemy.create_engine("mssql+pyodbc:///?odbc_connect={}".format(params_datahub))

In [0]:
# df=pd.read_csv(r"C:\Business_Data_Analysis\data\credit_train.csv")
# df=pd.read_sql_table(r"credit_train",engine_datahub)
# df.head()

df=pd.read_csv(r"/Workspace/Users/bdaconsulting1098@gmail.com/bda_course/BDA2/data/credit_train.csv")
df.head()

In [0]:
df['Loan Status'].unique()
# df['Years in current job'].unique()

## Exploratory Data Analysis(EDA)



### Check missing values and shape
Normally we need to clean the samples, i,e, impute missing values but in this case the data is pretty clean with no missing values. We also check the shape to make sure it matches the meta data info in the document. 

In [0]:
df.info(), df.shape

In [0]:
df.shape

### Check Loan Status ratio
The samples are balanced so we can use "Accuracy" metric to measure the performance of the model

In [0]:
df['Loan Status'].value_counts().plot(kind='bar')

### Describe the data

Categorical features/Dimensions 

In [0]:
cat_cols=df.select_dtypes(object).drop(['Loan ID','Customer ID'],axis=1).columns.tolist()
cat_cols

In [0]:
df['Term'].value_counts().plot(kind="bar")


In [0]:
# x = ('apple', 'banana', 'cherry')
# y = enumerate(x)

# for item in y:
#     print(item)

In [0]:
import matplotlib.pyplot as plt
categorical_features = cat_cols
# fig, ax = plt.subplots(1, len(categorical_features))
# fig, ax = plt.subplots(1, len(categorical_features))
for i, categorical_feature in enumerate(df[categorical_features]):
    print(i,categorical_feature)

In [0]:
import matplotlib.pyplot as plt
categorical_features = cat_cols
# fig, ax = plt.subplots(1, len(categorical_features))
fig, ax = plt.subplots(1, len(categorical_features))
for i, categorical_feature in enumerate(df[categorical_features]):
    df[categorical_feature].value_counts().plot(kind="bar", ax=ax[i],figsize=(20,8),rot=90,fontsize=10).set_title(categorical_feature)
fig.show()

Numeric data

#### Histogram
Histogram groups numeric data into bins, displaying the bins as segmented columns and summarize the distribution of a univariate data set. 

In [0]:
num_cols=df.select_dtypes('number').columns.tolist()
num_cols

In [0]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
fig, ax = plt.subplots(6, 2, figsize=(20, 20))

for i in range(ax.shape[0]):
    for j in range(ax.shape[1]):
        sns.distplot(df[df[num_cols].columns[i*2+j]], ax=ax[i][j],bins=50)

Boxplot : ![image.png](attachment:image.png)

In [0]:
import warnings
warnings.simplefilter(action='ignore', category=UserWarning)

fig, ax = plt.subplots(6, 2, figsize=(15, 25))

#plot the features except LOCATION_ID and Risk
for i in range(ax.shape[0]):
    for j in range(ax.shape[1]):
#         sns.boxplot(df[df[num_cols].columns[i*2+j]], ax=ax[i][j],orient='v',showfliers=False)
        sns.boxplot(df[df[num_cols].columns[i*2+j]], ax=ax[i][j],orient='v')

In [0]:
df.describe()
# from the "max" row we can see feature PARA_A, PARA_B,Money_Value,History have some potential outliers. 

In [0]:
df['Credit Score']=np.where(df['Credit Score']>=1000,df['Credit Score']/10,df['Credit Score'])

In [0]:
df['Current Loan Amount']=np.where(df['Current Loan Amount']==99999999,df['Current Loan Amount'].median(),df['Current Loan Amount'])

In [0]:
df.describe()

### Clip, i.e. assigns values outside boundary to boundary values, the data to deal with outliers. 
 Outliers may distort how we see the data. They contain information too so it's a tradeoff; we lose some info but gain a better big picture of the data.

In [0]:
df['Years in current job']=df['Years in current job'].str[:2].replace({'< ':'0.5'}).astype(float)

In [0]:
# here we use quantile 0.01 as lower limit and 0.99 upper.
df[num_cols]=df[num_cols].clip(lower=df[num_cols].quantile(0.01), upper=df[num_cols].quantile(0.99),axis=1)
df.describe()

In [0]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
fig, ax = plt.subplots(6, 2, figsize=(20, 25))

for i in range(ax.shape[0]):
    for j in range(ax.shape[1]):
        sns.distplot(df[df[num_cols].columns[i*2+j]], ax=ax[i][j],bins=50)

### Boxplot the data
Boxplot shows the shape of the distribution, its central value, and its variability

In [0]:
import warnings
warnings.simplefilter(action='ignore', category=UserWarning)

fig, ax = plt.subplots(6, 2, figsize=(15, 25))

#plot the features except LOCATION_ID and Risk
for i in range(ax.shape[0]):
    for j in range(ax.shape[1]):
        sns.boxplot(df[df[num_cols].columns[i*2+j]], ax=ax[i][j],orient='v',showfliers=False)

#### Explain the Boxplots:
The boxplots tell similar story as the Histograms. None of the distributions seem normal.

### Correlation heatmap
Correlation heatmap allows us to see relations between features/attributes. The higher the absolute coefficient, the stronger the correlation is. 

In [0]:
import seaborn as sns
df=df.query('`Loan ID`.notnull()', engine='python')
sns.pairplot(df.sample(frac=0.01, replace=True).reset_index(drop=True),plot_kws=dict(marker="+", linewidth=1))

In [0]:

# df['Loan Status encoded']=np.where(df['Loan Status']=='Fully Paid',0,1)
corrMatrix = df.corr()
plt.figure(figsize = (12,9))
ax=(sns.heatmap(corrMatrix, annot=True))
plt.show()

In [0]:
from scipy.stats import chisquare,chi2_contingency

v1=df['Term']
# v1=df['Purpose']
# v2=df['Home Ownership']
v2=df['Loan Status']
# g, p, dof, expctd=chi2_contingency(pd.crosstab(v1, v2))
print('Chi Square p value =' , chi2_contingency(pd.crosstab(v1, v2))[1])

display(pd.crosstab(v1, v2))
display(pd.crosstab(v1, v2,normalize='index'))
# sns.heatmap(pd.crosstab(v1, v2), annot=True)
sns.heatmap(pd.crosstab(v1, v2,normalize='index'), annot=True,cmap="Blues")


In [0]:
for cat_col in cat_cols:
    for num_col in num_cols:    
        df.groupby(cat_col)[num_col].mean().plot(kind='bar',title=num_col)
        plt.show()    


In [0]:
#Plot each attribute vs Class in percentage
y = 'Loan Status'
for i, predictor in enumerate(df[cat_cols].drop(columns=['Loan Status'])):
    plt.figure(i)
    df1 = df.groupby(predictor)[y].value_counts(normalize=True)
    df1 = df1.mul(100)
    df1 = df1.rename('percent').reset_index()

    g = sns.catplot(x=predictor,y='percent',hue=y,kind='bar',data=df1,
                    palette=sns.color_palette(['lightseagreen', 'tomato']),height=8.27, aspect=11.7/8.27)
    g.set_xticklabels(rotation=30)
    g.ax.set_ylim(0,100)

    for p in g.ax.patches:
        txt = str(np.nan_to_num(p.get_height().round(2))) + '%'
        txt_x = p.get_x() 
        txt_y = p.get_height()
        txt_y=np.nan_to_num(txt_y)
        g.ax.text(txt_x,txt_y,txt)

## Data processing and cleaning

In [0]:
df=pd.read_csv(r"/Workspace/Users/bdaconsulting1098@gmail.com/bda_course/BDA2/data/credit_train.csv")
df.head()

In [0]:
# limitPer = len(df) * .80
# df = df.dropna(thresh=limitPer, axis=1)

# limitPer=df.shape[1] * .50
# df = df.dropna(thresh=limitPer, axis=0)

df.shape
# df.head()

df['Years in current job']=df['Years in current job'].str[:2].replace({'< ':'0.5'}).astype(float)
num_cols=df.select_dtypes('number').columns.tolist()

cat_cols=df.select_dtypes(object).drop(['Loan ID','Customer ID'],axis=1).columns.tolist()

df['Credit Score']=np.where(df['Credit Score']>=1000,df['Credit Score']/10,df['Credit Score'])
df['Current Loan Amount']=np.where(df['Current Loan Amount']==99999999,df['Current Loan Amount'].median(),df['Current Loan Amount'])

df[num_cols]=df[num_cols].clip(lower=df[num_cols].quantile(0.01), upper=df[num_cols].quantile(0.99),axis=1)

# df.query('`Loan ID`.isnull()',engine='python')
df=df.query('`Loan ID`.notnull()',engine='python')
df

In [0]:
#replace mnissing value with median, a better representation of the center of the data if it's not normally ditributed

from sklearn.impute import SimpleImputer
imputer = SimpleImputer(missing_values=np.nan, strategy='median')
for col in num_cols:
    df[col]=imputer.fit_transform(df[col].values.reshape(-1, 1))



In [0]:
imputer = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
for col in cat_cols:
    df[col]=imputer.fit_transform(df[col].values.reshape(-1, 1))

In [0]:
c='Purpose'
df[c].value_counts()

In [0]:
c='Purpose'
percentage=5
dct={}
series = pd.value_counts(df[c])
mask = (series/series.sum() * 100).lt(percentage)             
dct[c] = series[mask].index.values.tolist()
dct[c]
df[c] = np.where(df[c].isin(dct[c]),'Other',df[c])
df[c].value_counts()

In [0]:
df.query('`Loan ID`.notnull()',engine='python')

In [0]:
#check for missing values
df[df.isnull().any(axis=1)]

In [0]:
#encode the attribute
def one_hot(df, cols):
    """
    @param df pandas DataFrame
    @param cols a list of columns to encode 
    @return a DataFrame with one-hot encoding
    """
    for each in cols:
        dummies = pd.get_dummies(df[each], prefix=each, drop_first=False)
        df = pd.concat([df, dummies], axis=1)
        df = df.drop([each], axis=1)
    return df


cat_cols.remove('Loan Status')
df=one_hot(df,cat_cols)
df.head()

In [0]:
item_list = df.columns.tolist()
item_list = [e for e in item_list if e not in ('Loan ID','Customer ID','Loan Status','Loan Status encoded')]
item_list

# Optional start --------------------------------------------

# Checking for multicollinearity using VIF(Variance Inflation Factor)

A variance inflation factor (VIF) is a measure of the amount of multicollinearity in regression analysis. Multicollinearity exists when there is a correlation between multiple independent variables in a multiple regression model. This can adversely affect the regression results. The higher the value, the greater the correlation of the variable with other variables. Values of more than 4 or 5 are sometimes regarded as being moderate to high, with values of 10 or more being regarded as very high.

In [0]:

def vif_cal(input_data, dependent_col):
    x_vars=input_data.drop([dependent_col], axis=1)
    xvar_names=x_vars.columns
    for i in range(0,xvar_names.shape[0]):
        y=x_vars[xvar_names[i]] 
        x=x_vars[xvar_names.drop(xvar_names[i])]
        rsq=sm.ols(formula="y~x", data=x_vars).fit().rsquared  
        vif=round(1/(1-rsq),2)
        print (xvar_names[i], " VIF = " , vif)

import statsmodels.formula.api as sm
vif_cal(df[item_list+['Loan Status']],'Loan Status')      

# Feature selection

In [0]:
y =df['Loan Status'].map({'Fully Paid':0,'Charged Off':1})# target variable
X= df[item_list]# features after dropping the  target (diagnosis) & ID
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(X, y, test_size=0.3, random_state=42)

### KBest

In [0]:
from sklearn.feature_selection import SelectKBest, chi2
X_5_best= SelectKBest(chi2, k=15).fit(x_train, y_train)
mask = X_5_best.get_support() #list of booleans for selected features
new_feat = [] 
for bool, feature in zip(mask, x_train.columns):
    if bool:
        new_feat.append(feature)
print('The best features are:{}'.format(new_feat)) # The list of your 5 best features

### RFECV

In [0]:
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.feature_selection import RFECV
# cv_estimator = RandomForestClassifier(random_state =42)
# cv_estimator.fit(X_train, Y_train)
# cv_selector = RFECV(cv_estimator,cv= 5, step=1,scoring='accuracy')
# cv_selector = cv_selector.fit(X_train, Y_train)
# rfecv_mask = cv_selector.get_support() #list of booleans
# rfecv_features = [] 
# for bool, feature in zip(rfecv_mask, X_train.columns):
#     if bool:
#         rfecv_features.append(feature)
# print('Optimal number of features :', cv_selector.n_features_)
# print('Best features :', rfecv_features)


In [0]:

# n_features = x_train.shape[1]
# plt.figure(figsize=(8,8))
# plt.barh(range(n_features), cv_estimator.feature_importances_, align='center') 
# plt.yticks(np.arange(n_features), x_train.columns.values) 
# plt.xlabel('Feature importance')
# plt.ylabel('Feature')
# plt.show()

## Algorithm selection
Sometimes it's difficult to know which algorithm to use to train the model because each algorithm has its pros and cons so we test mainstream algorithms and pick the one with the best 
performance.

In [0]:
from sklearn import model_selection
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
import warnings
warnings.filterwarnings("ignore")


from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df_x =pd.DataFrame(scaler.fit_transform(df[item_list]),columns=item_list)
                   
df['Loan Status encoded']=np.where(df['Loan Status']=='Fully Paid',0,1)                   
                   
# load dataset
x_train, x_test, y_train, y_test = train_test_split(df_x, df['Loan Status encoded'], 
                                                    test_size = .25)

X = x_train
Y = y_train

# prepare configuration for cross validation test harness
seed = 7
# prepare models
models = []
models.append(('LR', LogisticRegression()))
models.append(('LDA', LinearDiscriminantAnalysis()))
models.append(('KNN', KNeighborsClassifier()))
models.append(('CART', DecisionTreeClassifier()))
models.append(('NB', GaussianNB()))
# models.append(('SVM', SVC()))
# evaluate each model in turn
results = []
names = []
scoring = 'roc_auc'
for name, model in models:
    kfold = model_selection.KFold(n_splits=10)
    cv_results = model_selection.cross_val_score(model, X, Y, cv=kfold, scoring=scoring)
    results.append(cv_results)
    names.append(name)
    msg = "%s: %f (%f)" % (name, cv_results.mean(), cv_results.std())
    print(msg)
# boxplot algorithm comparison
fig = plt.figure()
fig.suptitle('Algorithm Comparison')
ax = fig.add_subplot(111)
plt.boxplot(results)
ax.set_xticklabels(names)
plt.show()

### Select the best algorithm and do Grid Search for the best Hyper Parameters.
From the above boxplot we can see that the attributes are very powerful predictors and all algorithms have high performance.
In this case we choose Logistic Regression due to it's better interpretability. To further improve the performance we do Grid Search
to find the best Hyper Parameters for Logistic Regression

In [0]:
# Grid search cross validation
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression


grid={"C":np.logspace(-3,3,7), "penalty":["l1","l2"]}# l1 lasso l2 ridge
logreg=LogisticRegression()
logreg_cv=GridSearchCV(logreg,grid,cv=5)
logreg_cv.fit(x_train,y_train)

print("tuned hpyerparameters :(best parameters) ",logreg_cv.best_params_)
print("accuracy :",logreg_cv.best_score_)

# Optional end --------------------------------------------

In [0]:
df

In [0]:
#For better performance use MinMaxScaler to scale and translates each feature individually such that it is in the given range on the training set, e.g. between zero and one.
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df_x =pd.DataFrame(scaler.fit_transform(df[item_list]),columns=item_list)

In [0]:
df_x

In [0]:
df_y=np.where(df['Loan Status']=='Fully Paid',0,1)   

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, precision_score, recall_score
from sklearn.metrics import precision_recall_curve, average_precision_score


                   
#splitting the principal training dataset to subtrain and subtest datasets
x_train, x_test, y_train, y_test = train_test_split(df_x, df_y, test_size = .3)

from sklearn.linear_model import LogisticRegression
logit = LogisticRegression()

logit.fit(x_train, y_train)
predictions = logit.predict(x_test)
probabilities = logit.predict_proba(x_test)
    
print('Algorithm:', type(logit).__name__)
print("\nClassification report:\n", classification_report(y_test, predictions))
print("Accuracy Score:", accuracy_score(y_test, predictions))

In [0]:
# ROC

In [0]:
#confusion matrix
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import plotly.offline as py
conf_matrix = confusion_matrix(y_test, predictions)

trace=go.Heatmap(z = conf_matrix,x = ["0", "1"],y = ["0", "1"],showscale = False, colorscale = "Picnic")
fig = make_subplots()
fig.add_trace(trace)
py.iplot(fig)

In [0]:
column_df = pd.DataFrame(x_train.columns.tolist())
coefficients = pd.DataFrame(logit.coef_.ravel())
coef_sumry = (pd.merge(coefficients, column_df, left_index=True, 
                               right_index=True, how="left"))
coef_sumry.columns = ["coefficients", "features"]
coef_sumry = coef_sumry.sort_values(by = "coefficients", ascending=False)
display(coef_sumry)
trace = go.Bar(x = coef_sumry["features"], y = coef_sumry["coefficients"])


fig = make_subplots()
fig.add_trace(trace)
py.iplot(fig)

### Interpret the results:
<!-- An Accuracy Score of 0.96 on Test data is a very good score with 1 being perfect 100% correct prediction. 
From the confusion Matrix we know that out of 194 predictions, only 7 mistake. Area under curve(True Positive/ False Positive), 
another model performance metric which often is used for unbalanced samples, is 0.965, also near perfect. 
The Feature Importance chart  suggests that  Money_Values, PARA_B, PARA_A, Score and District_Loss are more powerful predictors for Risk. 
 -->
<!-- Overall we have a very good model that can predict Risk. -->

# Productization of your Insights/Recommendations

In [0]:
df=pd.read_csv(r"/Workspace/Users/bdaconsulting1098@gmail.com/bda_course/BDA2/data/credit_train.csv")
df.head()


# limitPer = len(df) * .80
# df = df.dropna(thresh=limitPer, axis=1)

# limitPer=df.shape[1] * .50
# df = df.dropna(thresh=limitPer, axis=0)

df.shape

df['Years in current job']=df['Years in current job'].str[:2].replace({'< ':'0.5'}).astype(float)
num_cols=df.select_dtypes('number').columns.tolist()

cat_cols=df.select_dtypes(object).drop(['Loan ID','Customer ID'],axis=1).columns.tolist()

df['Credit Score']=np.where(df['Credit Score']>=1000,df['Credit Score']/10,df['Credit Score'])
df['Current Loan Amount']=np.where(df['Current Loan Amount']==99999999,df['Current Loan Amount'].median(),df['Current Loan Amount'])

df[num_cols]=df[num_cols].clip(lower=df[num_cols].quantile(0.01), upper=df[num_cols].quantile(0.99),axis=1)

# df.query('`Loan ID`.isnull()',engine='python')
df=df.query('`Loan ID`.notnull()',engine='python')

#replace mnissing value with median, a better representation of the center of the data if it's not normally ditributed

from sklearn.impute import SimpleImputer
imputer = SimpleImputer(missing_values=np.nan, strategy='median')
for col in num_cols:
    df[col]=imputer.fit_transform(df[col].values.reshape(-1, 1))



imputer = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
for col in cat_cols:
    df[col]=imputer.fit_transform(df[col].values.reshape(-1, 1))

c='Purpose'
percentage=5
dct={}
series = pd.value_counts(df[c])
mask = (series/series.sum() * 100).lt(percentage)             
dct[c] = series[mask].index.values.tolist()
dct[c]
df[c] = np.where(df[c].isin(dct[c]),'Other',df[c])
df[c].value_counts()

unique_counts = pd.DataFrame.from_records([(col, df[col].nunique()) for col in df.columns],
                          columns=['Column_Name', 'Num_Unique']).sort_values(by=['Num_Unique'])

#encode the attribute
def one_hot(df, cols):
    """
    @param df pandas DataFrame
    @param cols a list of columns to encode 
    @return a DataFrame with one-hot encoding
    """
    for each in cols:
        dummies = pd.get_dummies(df[each], prefix=each, drop_first=False)
        df = pd.concat([df, dummies], axis=1)
        df = df.drop([each], axis=1)
    return df


# cat_cols.remove('Loan Status')
df=one_hot(df,cat_cols)
df.head()

item_list = df.columns.tolist()
item_list = [e for e in item_list if e not in ('Loan ID','Customer ID','Loan Status','Loan Status encoded')]

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df_x =pd.DataFrame(scaler.fit_transform(df[item_list]),columns=item_list)
                   


In [0]:
df_x.drop(['Loan Status_Charged Off','Loan Status_Fully Paid'],axis=1,inplace=True)


In [0]:
df_x.drop(['Loan Status_Charged Off','Loan Status_Fully Paid'],axis=1,inplace=True)
df['probabily'] = logit.predict_proba(df_x)[:,1]
df.sort_values(by='probabily',ascending=False).head(50)


In [0]:
import pyodbc
import urllib
import sqlalchemy

'''connect to datahub'''

# params_datahub = urllib.parse.quote_plus("DRIVER={SQL Server Native Client 11.0};"
#                                  "SERVER=localhost\SQLEXPRESS;"
#                                  "DATABASE=datahub;"
#                                  "UID=sa;"
#                                  "PWD=user1")

# engine_datahub = sqlalchemy.create_engine("mssql+pyodbc:///?odbc_connect={}".format(params_datahub))
# df=df.drop('Purpose_other',axis=1)
# df.to_sql('credit_prediction', engine_datahub,if_exists='replace',)



In [0]:
df_spark=spark.createDataFrame(df)
df_spark.write.mode("overwrite").saveAsTable("credit_prediction")

df_spark.createOrReplaceTempView('course_db.credit_prediction')


In [0]:
%sql
select * from course_db.credit_prediction 